# Notebook 07 — Embedding Analysis & Model Comparison (BioBERT vs ClinicalBERT)

---

## Objectives

Compare the two candidate encoders' TRAIN embeddings only — never validation or test — on purely descriptive statistics: distribution, pairwise cosine similarity, PCA variance, and cross-model correlation.

> This notebook only ever loads `train_embeddings.npy` for both models (see Section 2) — validation and test embeddings are not touched here, so there is no risk of this comparison influencing model selection based on held-out data. As the conclusion notes, the two encoders look very similar on these statistics alone (cosine correlation ≈ 0.92), so the actual choice between them is deferred to downstream **fusion validation performance**, not to anything measured in this notebook.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

## 1. Setup


In [2]:
ROOT1 = Path("/kaggle/input/datasets/mariammohamed1095/biobert/datasets/processed/nlp")
ROOT2 = Path("/kaggle/input/datasets/mariammohamed1095/clinicalbert/datasets/processed/nlp")

## 2. Load TRAIN Embeddings Only (Both Models)


In [3]:
BIO_DIR = ROOT1 / "biobert-base-cased-v1.1"

CLINICAL_DIR = ROOT2 / "Bio_ClinicalBERT"

## 3. Distribution Statistics


In [4]:
bio_train = np.load(
    BIO_DIR / "train_embeddings.npy"
)

clinical_train = np.load(
    CLINICAL_DIR / "train_embeddings.npy"
)

print(bio_train.shape)
print(clinical_train.shape)

(257, 768)
(257, 768)


## 4. Pairwise Cosine Similarity (Within Each Model)


In [5]:
stats = pd.DataFrame({

    "Metric":[

        "Mean",

        "Std",

        "Min",

        "Max"

    ],

    "BioBERT":[

        bio_train.mean(),

        bio_train.std(),

        bio_train.min(),

        bio_train.max()

    ],

    "ClinicalBERT":[

        clinical_train.mean(),

        clinical_train.std(),

        clinical_train.min(),

        clinical_train.max()

    ]

})

stats

,Metric,BioBERT,ClinicalBERT
0,Mean,-0.000870,-0.000687
1,Std,0.036074,0.036078
2,Min,-0.875680,-0.851681
3,Max,0.084454,0.078588


## 5. PCA — Explained Variance


In [6]:
bio_similarity = cosine_similarity(
    bio_train
)

clinical_similarity = cosine_similarity(
    clinical_train
)

bio_mean = bio_similarity.mean()

clinical_mean = clinical_similarity.mean()

print("BioBERT :", bio_mean)

print("Clinical :", clinical_mean)

BioBERT : 0.98395574
Clinical : 0.9848986


### 5.1 Top-20 Component Variance Table


In [7]:
bio_pca = PCA(
    n_components=20
)

bio_pca.fit(
    bio_train
)

clinical_pca = PCA(
    n_components=20
)

clinical_pca.fit(
    clinical_train
)

PCA(n_components=20)

### 5.2 Total Variance Captured by 20 Components


In [8]:
comparison = pd.DataFrame({

    "Component":

    np.arange(1,21),

    "BioBERT":

    bio_pca.explained_variance_ratio_,

    "ClinicalBERT":

    clinical_pca.explained_variance_ratio_

})

comparison.head()

,Component,BioBERT,ClinicalBERT
0,1,0.115249,0.128225
1,2,0.066526,0.065468
2,3,0.053627,0.053535
3,4,0.048030,0.047649
4,5,0.040303,0.038027


## 6. Per-Feature Variance


In [9]:
print(

"Bio Total:",

bio_pca.explained_variance_ratio_.sum()

)

print(

"Clinical Total:",

clinical_pca.explained_variance_ratio_.sum()

)

Bio Total: 0.6428198
Clinical Total: 0.6437876


## 7. Cross-Model Correlation

Flattened correlation between BioBERT and ClinicalBERT train embeddings — a high value indicates the two encoders capture largely overlapping semantic information.


In [10]:
bio_feature_var = np.var(
    bio_train,
    axis=0
)

clinical_feature_var = np.var(
    clinical_train,
    axis=0
)

print(

bio_feature_var.mean()

)

print(

clinical_feature_var.mean()

)

2.089105e-05
1.9663272e-05


## 8. Summary Table


In [11]:
correlation = np.corrcoef(

bio_train.flatten(),

clinical_train.flatten()

)[0,1]

print(correlation)

0.9211317235855981


## 9. Conclusion


In [12]:
summary = pd.DataFrame({

"Metric":[

"Mean Cosine",

"Mean Variance",

"PCA Variance"

],

"BioBERT":[

bio_mean,

bio_feature_var.mean(),

bio_pca.explained_variance_ratio_.sum()

],

"ClinicalBERT":[

clinical_mean,

clinical_feature_var.mean(),

clinical_pca.explained_variance_ratio_.sum()

]

})

summary

,Metric,BioBERT,ClinicalBERT
0,Mean Cosine,0.983956,0.984899
1,Mean Variance,0.000021,0.000020
2,PCA Variance,0.642820,0.643788


The embedding statistics produced by BioBERT and Bio_ClinicalBERT are highly similar. Both models generate normalized and stable embeddings with comparable variance and PCA characteristics. The high correlation (≈0.92) indicates that both models capture largely overlapping semantic representations. Therefore, the final model selection should be based on downstream fusion performance rather than embedding statistics alone.